### install dependency


In [92]:
from IPython.display import display, Markdown
from langchain_ollama import ChatOllama
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
import pymupdf
import re

from youtube_transcript_api import YouTubeTranscriptApi

#### 2. defining the schema

In [93]:
from pydantic import BaseModel, Field
from typing import Annotated, List

class Schema(BaseModel):
    technical_skill : Annotated[List[str], Field(description="list of skill which can be present in a resume, from the given resume content list")]
    # non_technical_skill : Annotated[List[str], Field(description="list of keyword can't be count in skill, from the given resume content list")]
    

### 3. defining model

In [94]:
model = ChatOllama(
    model="qwen3:8b"
)

embeding_model = ChatOllama(
    model="qwen3-embedding:4b"
)

structured_model = model.with_structured_output(schema=Schema)

# 4. Data Gather

### 4.1 defining YT_Video Script extractor (gathering data from podcast)

In [95]:
ytt_api = YouTubeTranscriptApi()
video_id = "YL0xu8Z73v4"

FetchedTranscript = ytt_api.fetch(
    video_id = video_id,
    languages = ["en"]
)

transcriptData = "".join(snippet.text for snippet in FetchedTranscript)

In [ ]:
from IPython.display import display, Markdown

display(Markdown(transcriptData))

#### 4.1.1 defining Vector DataBase

In [104]:
Vector_DB = Chroma(
    embedding_function = embeding_model,
    persist_directory = "Chroma_DB",
    collection_name = "YT_Podcast"
)

#### 4.1.2 defining Text Spliter


In [151]:
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

In [152]:
splited_text = text_spliter.split_text(text=transcriptData)

In [ ]:
splited_text[0]

In [ ]:
chunk_number = 1
docs = []
for chunk in splited_text:
    docs.append(Document(page_content=chunk, metadata={"chunk_number":chunk_number}))
    chunk_number += 1

In [ ]:
docs

In [157]:
chunk_summary_templete = PromptTemplate(
    template= """
You are summarizing a YouTube video transcript.

Summarize the following section clearly and concisely.

Focus on:
- Main ideas
- Important concepts
- Technical concepts
- Examples
- Important conclusions

Do not add information that is not present in the transcript.

Chunk_Content:
{Chunk_Content}
""",
input_variables= ["chunk_Content"]
)

In [158]:
chunk_summary_prompt =  chunk_summary_templete.invoke({"Chunk_Content":splited_text[0]})
chunk_summary_response = model.invoke(chunk_summary_prompt)

In [ ]:
# create a program for summary generation
summary = 


In [ ]:
summaries = []

for chunk in splited_text:
    chunk_summary_prompt =  chunk_summary_templete.invoke({"Chunk_Content":chunk})
    chunk_summary_response = model.invoke(chunk_summary_prompt)
    summaries.append(chunk_summary_response)

### gathering data from job platform

### gathering data from web

### gathering data from company hiring trend

### Extracting the skill from the top and trending projects in market

# Finding Gaps

### Skill gap in compare to JD

#### Defining PDF reader

In [49]:
pdf_text = pymupdf.open("ARV_cv.pdf",)

In [50]:
pdf_data = ""
for page in pdf_text:
    pdf_data = pdf_data + page.get_text()

pdf_data = pdf_data.lower().strip()

In [52]:
# removing URL or link
pdf_data = re.sub(r'https?://\S+|www\.\S+|\n',"",pdf_data)

# removing newline 
pdf_data = re.sub(r'\s*\n\s*', ' ', pdf_data)
pdf_data = pdf_data.split(" ")


In [53]:
extracted_preprocessed_skill_templete = PromptTemplate(
    template= """
remove the item from the list which can't be considered as technical skill for resume.

list:
{pdf_data}
""", 
input_variables= ["pdf_data"]
)

In [54]:
extracted_preprocessed_skill_prompt = extracted_preprocessed_skill_templete.invoke({"pdf_data":pdf_data})

In [57]:
response =  structured_model.invoke(extracted_preprocessed_skill_prompt)

In [58]:
technical_skill = response.technical_skill
non_technical_skill = response.non_technical_skill

In [ ]:
count = 1
for i in technical_skill:
    display(Markdown(i))
    count += 1

### Skill gap in compare to Field

### Skill gap required to join a specific company